In [1]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import pickle
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from dotenv import load_dotenv
load_dotenv()
#openai.api_key = os.getenv("OPENAI_KEY")

True

In [3]:
# client = OpenAI()
# defaults to getting the key using os.environ.get("OPENAI_KEY")
# if you saved the key under a different environment variable name, you can do something like:
from openai import OpenAI
client = OpenAI(
api_key=os.environ.get("OPENAI_KEY"),
)

In [20]:
# Make a request to the OpenAI API
#try:
#    response = client.chat.completions.create(
#      model="gpt-4",
#      messages=[
#        {"role": "system", "content": "You are a poetic assistant, skilled in explaining love to a wakanda girl."},
#        {"role": "user", "content": "Compose a poem in spanish that explains the concept of discipline."}
#      ],
#      max_tokens=60
#    )
#    print(response.choices[0].message)
#except Exception as e:
#    print(f"An error occurred: {e}")

ChatCompletionMessage(content='Disciplina, brillante como la estrella nocturna,\nUn rito sagrado que en nuestras vidas se encarna.\nNo es pura restricción, ni deseo de controlar,\nEs saber cuando luchar, es aprender a esperar.\n\nEs la hoja que for', role='assistant', function_call=None, tool_calls=None)


In [4]:
np.random.seed(42)
random_state=42

## Setting Up Functions

In [5]:
df_info = pd.read_csv('dataset_info.csv')
df_info.head(2)

,Dataset Name,Note,Study Title,Type,Topic,# Abstracts,# Included Abstracts,Inclusion Criteria,Exclusion Criteria
0,scopus_wosNullDuplicated.csv,Duplicates removed,NaN,Journal articles and conference papers,NaN,1376,48,1. Must show the capabilities of the artificia...,1. Studies in which there are no artificial ag...


In [8]:
def return_criteria(df, instructions, inclusion_criteria, exclusion_criteria, show_prompt=False):
    title = df['title']
    abstract = df['abstract']
    authorKeywords = df['author(s)Keywords']
    indexKeywords = df['indexKeywords']
    keywordsPlus = df['keywordsPlus']
    year = df['year']
    documentType = df['documentType']
    prompt = f'{instructions}\n\nTitle: {title}\n\nAbstract:\n{abstract}\n\nKeywords:\n{authorKeywords}, {indexKeywords}, {keywordsPlus}\n\nYear:\n{year}\n\nDocument Type:\n{documentType}\n\nInclusion criteria:\n{inclusion_criteria}\n\nExclusion criteria:\n{exclusion_criteria}\n\nDecision:',
    
    if show_prompt:
        print(prompt[0])

    memory=[{"role": "user", "content": prompt[0]}]
    try:
        response = client.chat.completions.create(
            model='gpt-4',
            messages=memory,
            temperature=0,
            top_p=0.5,
        )
    except:
        print('Context length exceeded, attempting with GPT-4-TURBO.')
        response = client.chat.completions.create(
            model='gpt-4-turbo',
            messages=memory,
            temperature=0,
            top_p=0.5,
        )
        print('Successfully completed with GPT-4-TURBO.')
    return response.choices[0].message.content.strip('.').lower().capitalize()
    #return response['choices'][0]['message']['content'].strip('.').lower().capitalize()

In [19]:
def loop_through_df(df, instructions, save_ckpt=True, sample=False, sample_n=3, random_state=42):
    dfs = dict()
    for ind, row in df.iterrows():
        print(f'Running: {row["Dataset Name"]}')
        if sample:
            temp_df = pd.read_csv(f"data/{row['Dataset Name']}", encoding='ISO-8859-1').sample(sample_n, random_state=random_state)
        else:
            temp_df = pd.read_csv(f"data/{row['Dataset Name']}", encoding='ISO-8859-1')
        inclusion_criteria = row['Inclusion Criteria']
        exclusion_criteria = row['Exclusion Criteria']
        try:
            temp_df['decision'] = temp_df.apply(lambda x: return_criteria(x, instructions, inclusion_criteria, exclusion_criteria), axis=1)
        except Exception as e:
            print(f"Error processing file: {e}")
            print(f'Error processing {row["Dataset Name"]}')
            continue
        temp_df['decision'] = temp_df['decision'].apply(
            lambda x: "Included" if "Included" in x else "Excluded" if "Excluded" in x else "Undecided")
        dfs[row['Dataset Name']] = temp_df
        # save checkpoint
        if save_ckpt:
            temp_df.to_csv(f'results/{row["Dataset Name"]}')
            with open(f'checkpoints/{row["Dataset Name"]}.pkl', 'wb') as f:
                pickle.dump(temp_df, f)
    return dfs

## [CAUTION: CALLING API] Running Sampled Analysis

In [20]:
instructions = \
"""Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else."""
print(instructions)

Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else.


In [21]:
#sampled_dfs = loop_through_df(df_info, instructions, save_ckpt=False, sample=True, sample_n=2)

Running: scopus_wosNullDuplicated.csv


In [31]:
#for k, v in sampled_dfs.items():
#    print(k)
#    print(v)
#    print('='*20)

wosNullDuplicated_conference_papers.csv
                                                title  \
10  COMPETITIVE INTELLIGENCE IN THE DIGITAL AGE: N...   
9   Improving the Generalizability of Collaborativ...   

                                             abstract  \
10  The early period of Competitive Intelligence -...   
9   Conflict prediction in communication is integr...   

                                    author(s)Keywords indexKeywords  \
10  Competitive Intelligence; Digital Age; neurote...             *   
9                                                 NaN             *   

   keywordsPlus  year      documentType       tag  decision  
10          NaN  2019  Conference paper  Excluded  Excluded  
9           NaN  2023  Conference paper  Excluded  Excluded  


## Running On Entire Dataset

In [35]:
#CAUTION: Expensive and time-consuming call to the GPT API
#df_final = loop_through_df(df_info, instructions, save_ckpt=True, sample=False)

Running: wosNullDuplicated_conference_papers.csv


In [36]:
#for key, value in df_final.items():
#    value.to_csv(f'results/scopus_result.csv', index=False)

## Analysis

### Helper Functions

In [1]:
import pandas as pd

In [2]:
def get_absolute_accuracy(df):
    return df[df['tag'] == df['decision']].shape[0] / df.shape[0]

In [3]:
from sklearn.metrics import classification_report
def get_classification_report(df):
    return classification_report(df['tag'], df['decision'])

In [4]:
from sklearn.metrics import recall_score
def get_recall_score(df, tag):
    return recall_score(df['tag'], df['decision'], pos_label=tag)

In [5]:
def replace_undecided(df):
    df['decision'] = df['decision'].apply(lambda x: 'Excluded' if x == 'Undecided' else x)
    df['decision'] = df['decision'].apply(lambda x: x.capitalize())
    df['tag'] = df['tag'].apply(lambda x: x.capitalize())
    df['decision'] = df['decision'].apply(lambda x: "Included" if "Included" in x else "Excluded" if "Excluded" in x else "Undecided")
    return df

### Analyzing

In [6]:
df_wos = replace_undecided(pd.read_csv('results/wosNullDuplicated_conference_papers_results.csv'))

In [7]:
print(get_absolute_accuracy(df_wos))

1.0


In [109]:
# # very verbose
# print(get_classification_report(df_ivm))
# print(get_classification_report(df_lpvr))
# print(get_classification_report(df_ssri))
# print(get_classification_report(df_llm))

In [8]:
print(get_recall_score(df_wos, 'Included'))

1.0


In [9]:
print(get_recall_score(df_wos, 'Excluded'))

1.0


In [10]:
# combine all dfs
df_all = pd.concat([df_wos], ignore_index=True)

print('='*55)
print(f"Absolute accuracy: {get_absolute_accuracy(df_all)}")
print(f"Recall for Included: {get_recall_score(df_all, 'Included')}")
print(f"Recall for Excluded: {get_recall_score(df_all, 'Excluded')}")
print('='*55)
print(get_classification_report(df_all))
print('='*55)

Absolute accuracy: 1.0
Recall for Included: 1.0
Recall for Excluded: 1.0
              precision    recall  f1-score   support

    Excluded       1.00      1.00      1.00        11
    Included       1.00      1.00      1.00         1

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



In [11]:
df_all.shape[0]

12

## Subset Analysis

In [12]:
df_wos.iloc[0:10]

,title,abstract,author(s)Keywords,indexKeywords,keywordsPlus,year,documentType,tag,decision
0,INTELLIGENT TUTORING SYSTEMS - TRENDS ON DESIG...,Intelligent tutoring systems (ITS) are compute...,intelligent tutoring systems (ITS); principles...,*,COGNITIVE TUTORS,2016,Conference paper,Excluded,Excluded
1,AN AFFECTIVE INTELLIGENT TUTORING SYSTEM IN TH...,Autism spectrum disorder (ASD) is a neurologic...,Autism; affective intelligent tutoring systems...,*,CHILDREN,2017,Conference paper,Included,Included
2,"ACCESSIBILITY, BIASES AND ETHICS IN CHATBOTS A...",The use of chatbots is increasingly common in ...,Conversational agents; Chatbot; accessibility;...,*,NaN,2019,Conference paper,Excluded,Excluded
3,Effortlessly Populating Immersive Training Sim...,Immersive training simulations engage users in...,Immersive training; Virtual agents; Informed e...,*,NaN,2023,Conference paper,Excluded,Excluded
4,Detecting Early Signs of Depression in the Con...,The high prevalence of depression in society h...,Depression detection; Conversational domain; T...,*,EMOTION,2022,Conference paper,Excluded,Excluded
5,Modeling Learners' Cognitive and Affective Sta...,The relationship between learners' cognitive a...,Cognitive and affective states; self-regulated...,*,SELF-REGULATION; PSYCHOLOGICAL CONSTRUCTION; M...,2018,Conference paper,Excluded,Excluded
6,"3D CHATBOT IN HIGHER EDUCATION, HELPING STUDEN...",In our bachelor degree (Digital Arts and Enter...,coaching; chatbot; 3D,*,NaN,2019,Conference paper,Excluded,Excluded
7,Scripted Vicarious Dialogues: Educational Vide...,Videos are convenient resources for asynchrono...,video-based learning; augmented learning envir...,*,COGNITIVE ENGAGEMENT; CLASSROOM; ATTENTION; FR...,2023,Conference paper,Excluded,Excluded
8,User Involvement in Training Smart Home Agents,Smart home systems contain plenty of features ...,human-agent interaction; smart homes; supervis...,*,NaN,2022,Conference paper,Excluded,Excluded
9,Improving the Generalizability of Collaborativ...,Conflict prediction in communication is integr...,NaN,*,NaN,2023,Conference paper,Excluded,Excluded


In [13]:
# correctly excluded second row from df_wos
print(f"Title: {df_wos['title'][2]}")
print(f"Abstract: {df_wos['abstract'][2]}")
print(f"Author Keywords: {df_wos['author(s)Keywords'][2]}")
print(f"Index Keywords: {df_wos['indexKeywords'][2]}")
print(f"Keywords Plus: {df_wos['keywordsPlus'][2]}")
print(f"Year: {df_wos['year'][2]}")
print(f"Document Type: {df_wos['documentType'][2]}")
print(f"Human: {df_wos['tag'][2]}")
print(f"GPT: {df_wos['decision'][2]}")

Title: ACCESSIBILITY, BIASES AND ETHICS IN CHATBOTS AND INTELLIGENT AGENTS FOR EDUCATION
Abstract: The use of chatbots is increasingly common in universities around the world to support both educational and administrative tasks. There is also a growing awareness of the importance of inclusive education and, as a consequence, the need to comply with accessibility requirements regarding web contents and web interface (WCAG). These requirements are also supported by specific laws in most countries. But such awareness does not yet reach the interfaces of conversational agents or chatbots that are being developed. Within the framework of the European project ACACIA, co-funded by the Erasmus+ program of the European Union, Artemisa, a chatbot has been created dedicated to fight against sexual harassment and search for volunteers to promote the acceptance of diversity and tolerance, has been created using a platform that facilitates the generation and management of this type of artificial int

In [14]:
# correctly included 5th row of df_wos
print(f"Title: {df_wos['title'][1]}")
print(f"Abstract: {df_wos['abstract'][1]}")
print(f"Author Keywords: {df_wos['author(s)Keywords'][1]}")
print(f"Index Keywords: {df_wos['indexKeywords'][1]}")
print(f"Keywords Plus: {df_wos['keywordsPlus'][1]}")
print(f"Year: {df_wos['year'][1]}")
print(f"Document Type: {df_wos['documentType'][1]}")
print(f"Human: {df_wos['tag'][1]}")
print(f"GPT: {df_wos['decision'][1]}")

Title: AN AFFECTIVE INTELLIGENT TUTORING SYSTEM IN THE SPECIAL EDUCATION OF INDIVIDUALS WITH AUTISM
Abstract: Autism spectrum disorder (ASD) is a neurological disorder affecting the way in which the brain processes information. It can affect all aspects of a person's development. Autism is characterized by impairments in learning and communication, in social interaction, in imaginative ability, as well as, in repetitive and restricted patterns of behavior [1]. This research contributes to the advancement of Intelligent Tutoring Systems by proposing an affective intelligent tutoring system to be used in the field of specialized education in order to overcome the lack of individualized intervention as pertains to special education for individuals with learning disabilities such as autism. The Integrated Specialized Learning Application (ISLA) is an Affective Intelligent Tutoring System (AITS) that evolves along with the learner's needs. ISLA is unique and its contribution entails the com

In [15]:
# incorrectly excluded 31th row of df_wos
print(f"Title: {df_wos['title'][4]}")
print(f"Abstract: {df_wos['abstract'][4]}")
print(f"Author Keywords: {df_wos['author(s)Keywords'][4]}")
print(f"Index Keywords: {df_wos['indexKeywords'][4]}")
print(f"Keywords Plus: {df_wos['keywordsPlus'][4]}")
print(f"Year: {df_wos['year'][4]}")
print(f"Document Type: {df_wos['documentType'][4]}")
print(f"Human: {df_wos['tag'][4]}")
print(f"GPT: {df_wos['decision'][4]}")

Title: Detecting Early Signs of Depression in the Conversational Domain: The Role of Transfer Learning in Low-Resource Scenarios
Abstract: The high prevalence of depression in society has given rise to the need for new digital tools to assist in its early detection. To this end, existing research has mainly focused on detecting depression in the domain of social media, where there is a sufficient amount of data. However, with the rise of conversational agents like Siri or Alexa, the conversational domain is becoming more critical. Unfortunately, there is a lack of data in the conversational domain. We perform a study focusing on domain adaptation from social media to the conversational domain. Our approach mainly exploits the linguistic information preserved in the vector representation of text. We describe transfer learning techniques to classify users who suffer from early signs of depression with high recall. We achieve state-of-the-art results on a commonly used conversational data

In [16]:
# incorrectly included 1st row of df_wos
print(f"Title: {df_wos['title'][1]}")
print(f"Abstract: {df_wos['abstract'][1]}")
print(f"Author Keywords: {df_wos['author(s)Keywords'][1]}")
print(f"Index Keywords: {df_wos['indexKeywords'][1]}")
print(f"Keywords Plus: {df_wos['keywordsPlus'][1]}")
print(f"Year: {df_wos['year'][1]}")
print(f"Document Type: {df_wos['documentType'][1]}")
print(f"Human: {df_wos['tag'][1]}")
print(f"GPT: {df_wos['decision'][1]}")

Title: AN AFFECTIVE INTELLIGENT TUTORING SYSTEM IN THE SPECIAL EDUCATION OF INDIVIDUALS WITH AUTISM
Abstract: Autism spectrum disorder (ASD) is a neurological disorder affecting the way in which the brain processes information. It can affect all aspects of a person's development. Autism is characterized by impairments in learning and communication, in social interaction, in imaginative ability, as well as, in repetitive and restricted patterns of behavior [1]. This research contributes to the advancement of Intelligent Tutoring Systems by proposing an affective intelligent tutoring system to be used in the field of specialized education in order to overcome the lack of individualized intervention as pertains to special education for individuals with learning disabilities such as autism. The Integrated Specialized Learning Application (ISLA) is an Affective Intelligent Tutoring System (AITS) that evolves along with the learner's needs. ISLA is unique and its contribution entails the com

## Reasoning Analysis

### Explanation of Decision

In [21]:
def get_gpt_call(prompt):
    response = client.chat.completions.create(
            model='gpt-4',
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            top_p=0.5,
        )
    return response.choices[0].message.content

In [28]:
def create_prompt(df, row_num):
    prompt = f"""Explain your reasoning for the decision given with the information below.

Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else.

Title: {df['title'][row_num]}

Abstract:
{df['abstract'][row_num]}

Keywords:
{df['author(s)Keywords'][row_num]}, {df['indexKeywords'][row_num]}, {df['keywordsPlus'][row_num]}

Year:
{df['year'][row_num]}

Document Type:
{df['documentType'][row_num]}

Inclusion criteria:
1. Must show the capabilities of the artificial agent to support the students' emotional regulation
2. Must be empirical studies (quantitative, qualitative, or mixed) and primary data sources, that is, the studies must measure the effectiveness of the effect generated by the artificial agent, specifically its emotional learning strategies, in the learning outcomes variables
3. Must be applied in the context of computer-based learning. That is, a learning environment in which teaching and learning processes, synchronous and/or asynchronous, academic or auxiliary systems, are carried out with the help of computers. It is also accepted as a combination of computer-based learning environments with others, but this environment must be primary in implementing the artificial agent
4. Must be papers published from January 1, 2015, to February 16, 2024
5. Must be written in English
6. Must be peer-reviewed papers published in journals or conference proceedings.
7. Must be the most recent published papers on the emotion regulation strategies of artificial agents

Exclusion criteria:
1. Studies in which there are no artificial agents, use the Wizard of Oz technique, or these do not have the empathic capability (i.e., provide feedback based on the student's emotional state), which include emotion detection and support in the student's emotion regulation
2. Non-empirical studies and non-primary data sources, that is, studies that do not provide qualitative, quantitative, or mixed information on the effectiveness of the emotion regulation strategies of the artificial agents in the learning outcomes variables
3. Studies are not applied in computer-based learning. That is, the study does not show the implementation of the artificial agent in the process of synchronous and/or asynchronous computer-based teaching and learning, whether academic or auxiliary system
4. Studies published before 2015
5. Studies in other languages different from English
6. Studies published as book chapters, surveys, and/or reviews
7. Studies presenting artificial agents, specifically the same emotion regulation strategies, discussed in more recent papers

Decision: {df['decision'][row_num]}
"""
    return prompt

In [25]:
# correctly excluded second row from df_scopus
print(get_gpt_call(create_prompt(df_wos, 2)))

The article does not meet the first inclusion criteria as it does not show the capabilities of the artificial agent to support the students' emotional regulation. It also does not meet the second inclusion criteria as it is not an empirical study measuring the effectiveness of the effect generated by the artificial agent, specifically its emotional learning strategies, in the learning outcomes variables.


In [29]:
# correctly included 5th row of df_scopus
print(get_gpt_call(create_prompt(df_wos, 1)))

Included


In [ ]:
# incorrectly excluded 4th row of df_scopus
print(get_gpt_call(create_prompt(df_wos, 4)))

Reasoning: The study is not a randomized controlled trial or a non-randomized comparative cohort study, which is an inclusion criteria. Therefore, the article does not meet all inclusion criteria and must be excluded.


In [ ]:
# incorrectly included 1st row of df_scopus
print(get_gpt_call(create_prompt(df_wos, 1)))

The article meets all of the inclusion criteria and none of the exclusion criteria, therefore it should be included in the review paper.


### Reflection of Incorrect Answers

In [30]:
def reflect(df, row_num):
    prompt = f"""Explain your reasoning for why the decision given was incorrect with the information below.

Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else.

Title: {df['title'][row_num]}

Abstract:
{df['abstract'][row_num]}

Keywords:
{df['author(s)Keywords'][row_num]}, {df['indexKeywords'][row_num]}, {df['keywordsPlus'][row_num]}

Year:
{df['year'][row_num]}

Document Type:
{df['documentType'][row_num]}

Inclusion criteria:
1. Must show the capabilities of the artificial agent to support the students' emotional regulation
2. Must be empirical studies (quantitative, qualitative, or mixed) and primary data sources, that is, the studies must measure the effectiveness of the effect generated by the artificial agent, specifically its emotional learning strategies, in the learning outcomes variables
3. Must be applied in the context of computer-based learning. That is, a learning environment in which teaching and learning processes, synchronous and/or asynchronous, academic or auxiliary systems, are carried out with the help of computers. It is also accepted as a combination of computer-based learning environments with others, but this environment must be primary in implementing the artificial agent
4. Must be papers published from January 1, 2015, to February 16, 2024
5. Must be written in English
6. Must be peer-reviewed papers published in journals or conference proceedings.
7. Must be the most recent published papers on the emotion regulation strategies of artificial agents

Exclusion criteria:
1. Studies in which there are no artificial agents, use the Wizard of Oz technique, or these do not have the empathic capability (i.e., provide feedback based on the student's emotional state), which include emotion detection and support in the student's emotion regulation
2. Non-empirical studies and non-primary data sources, that is, studies that do not provide qualitative, quantitative, or mixed information on the effectiveness of the emotion regulation strategies of the artificial agents in the learning outcomes variables
3. Studies are not applied in computer-based learning. That is, the study does not show the implementation of the artificial agent in the process of synchronous and/or asynchronous computer-based teaching and learning, whether academic or auxiliary system
4. Studies published before 2015
5. Studies in other languages different from English
6. Studies published as book chapters, surveys, and/or reviews
7. Studies presenting artificial agents, specifically the same emotion regulation strategies, discussed in more recent papers

Decision: {df['decision'][row_num]}
"""
    return prompt

In [31]:
# incorrectly excluded 4th row of df_wos
print(get_gpt_call(reflect(df_wos, 4)))

The decision to exclude the article is correct. The paper does not meet the first inclusion criterion, which states that the paper must show the capabilities of the artificial agent to support the students' emotional regulation. The paper focuses on detecting early signs of depression in the conversational domain, not on supporting students' emotional regulation. Furthermore, the paper does not meet the second inclusion criterion, as it does not measure the effectiveness of the effect generated by the artificial agent, specifically its emotional learning strategies, in the learning outcomes variables. The paper also does not meet the third inclusion criterion, as it is not applied in the context of computer-based learning. Therefore, the decision to exclude the article is correct.


In [ ]:
# incorrectly included 1st row of df_wos
print(get_gpt_call(reflect(df_wos, 1)))

As per the given inclusion and exclusion criteria, the decision given is incorrect. The abstract does not mention the use of ivermectin in any of the studies. Therefore, the article cannot be included based on the inclusion criteria.
